In [5]:
import numpy as np
import pandas as pd
import your
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import sys
import matplotlib as mpl
from matplotlib import colors
from scipy.optimize import curve_fit, leastsq, minimize_scalar
import numpy.ma as ma
import matplotlib.ticker as ticker
from matplotlib.ticker import MaxNLocator
from astropy.time import Time
from typing import Union, List
from scipy.signal import find_peaks

import frb_functions

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
from matplotlib.ticker import ScalarFormatter

In [6]:
def load_fil(burst,burst_tstamp, downsamp_t=1,downsamp_f=1):

    burst = your.Your(burst)

    header_dict = frb_functions.your_header_to_dict(burst)
    tsamp = header_dict["tsamp"]
    nbins = header_dict["nspectra"]
    fsamp = header_dict["foff"] * -1
    freqs = header_dict["freqs"] # in Mhz
    
    tmin = 0.0
    tmax = nbins * burst.tsamp # seconds
    fmin = header_dict["fmin"]
    fmax = header_dict["fmax"]
    extent_b = [tmin, tmax, fmin, fmax]

    offset = burst_tstamp #
    extent_b[0] = (tmin - offset) * 1000
    extent_b[1] = (tmax - offset) * 1000
    print(f"the extent of the data is: {extent_b}")
    
    #time increasing, freq decreasing
    print("loading in the data")
    d = frb_functions.get_data(burst, nstart=0, nsamp=nbins, npoln=1)

    sides_bandpass = 'right' 
    scale_bandpass = 3

    #Correcting the bandpass
    stokes_i = frb_functions.correct_bandpass(d, sides=sides_bandpass, scale=scale_bandpass)
    
    print(f"downsample factors are: time:{downsamp_t} and freq:{downsamp_f}")
    if downsamp_f > 1 or downsamp_t > 1:
        print(f"running downsampling and correcting bandpass")
        stokes_i = frb_functions.downsample(stokes_i, freqf=downsamp_f, timef=downsamp_t)
        tsamp = tsamp * downsamp_t
        fsamp = fsamp * downsamp_f
        stokes_i = frb_functions.correct_bandpass(stokes_i, sides=sides_bandpass, scale=scale_bandpass)
        downsamp_flag = True
    else:
        downsamp_flag = False

    #Masking Subband edges
    print(f"Masking subband edges")
    nchans = header_dict["nchans"]
    bw = header_dict["bw"] * -1
    print(f"channel numbers: {nchans}")
    nchans_n = nchans/downsamp_f
    subband_edges_arr = frb_functions.get_subband_edges(bandwidth=bw, n_chan=int(nchans/downsamp_f))
    stokes_i = frb_functions.mask_array(stokes_i, subband_edges_arr)     

    new_quantiles = np.nanpercentile(stokes_i.compressed(), (1.0, 99.99))
        
    #Create x grid to plot the step function. 
    x_grid = np.linspace(extent_b[0], extent_b[1], endpoint=False, num=stokes_i.shape[1])

    prof = np.ma.mean(stokes_i,axis=0)
    stokes_i_normprof = frb_functions.normalize_1d(prof, sides=sides_bandpass, scale=scale_bandpass)

    #new_quantiles = np.nanpercentile(stokes_i, (1.0, 99.99))
    
    return stokes_i, stokes_i_normprof, x_grid, new_quantiles, extent_b, tsamp, fsamp

In [19]:
def check_offpulse_contamination(x_grid, burst_tstamp_ms, prof, 
                                  sides='right', scale=3, 
                                  burst_window_ms=50.0):
    """
    Check whether the assumed off-pulse region overlaps with the burst.

    Parameters
    ----------
    x_grid           : 1-D array   time axis [ms], already offset by burst_tstamp
    burst_tstamp_ms  : float       burst centre in the SAME units as x_grid [ms]
                                   (typically 0.0 if x_grid is already centred)
    prof             : 1-D array   normalised profile (for diagnostic plot)
    sides            : str         which side is used as off-pulse ('right','left','both')
    scale            : int         1/scale fraction used as off-pulse region
    burst_window_ms  : float       protected window around burst centre [ms]

    Returns
    -------
    contaminated : bool   True if overlap detected
    """
    nsamp = len(x_grid)
    tmin, tmax = x_grid[0], x_grid[-1]
    total_ms = tmax - tmin

    # Off-pulse region boundaries in time [ms]
    off_regions = []
    if sides in ('left', 'both'):
        off_regions.append((tmin, tmin + total_ms / scale))
    if sides in ('right', 'both'):
        off_regions.append((tmax - total_ms / scale, tmax))

    # Burst protection window
    burst_lo = burst_tstamp_ms - burst_window_ms / 2
    burst_hi = burst_tstamp_ms + burst_window_ms / 2

    contaminated = False
    for (off_lo, off_hi) in off_regions:
        overlap = max(0, min(burst_hi, off_hi) - max(burst_lo, off_lo))
        if overlap > 0:
            contaminated = True
            print(f"  WARNING: burst window [{burst_lo:.1f}, {burst_hi:.1f}] ms "
                  f"overlaps off-pulse region [{off_lo:.1f}, {off_hi:.1f}] ms "
                  f"by {overlap:.1f} ms")

    if not contaminated:
        print(f"  OK: burst window [{burst_lo:.1f}, {burst_hi:.1f}] ms "
              f"does not overlap any off-pulse region.")

    return contaminated

In [9]:
def radiometer(tsamp, bw, npol, SEFD):
    """
    Flux density per unit S/N, from the radiometer equation.

    Parameters
    ----------
    tsamp : float   time resolution  [ms]
    bw    : float   bandwidth        [MHz]
    npol  : int     number of polarisations
    SEFD  : float   System-Equivalent Flux Density [Jy]
                    SEFD = Tsys / G  (Effelsberg L-band ≈ 13 Jy)

    Returns
    -------
    float : flux density per unit S/N  [Jy]
    """
    print("  radiometer inputs:")
    print(f"    SEFD : {SEFD} Jy")
    print(f"    bw   : {bw} MHz")
    print(f"    npol : {npol}")
    print(f"    tsamp: {tsamp} ms")
    return SEFD / np.sqrt(bw * 1e6 * npol * tsamp * 1e-3)

In [17]:
def flux_to_Lv(S_v_Jy, distance_Mpc, z):
    """
    Spectral luminosity from peak flux density.

    Convention:
        L_v = [ 4 * pi * D_L^2 / (1 + z) ] * S_v

    Parameters
    ----------
    S_v_Jy      : float   peak flux density      [Jy]
    distance_Mpc: float   luminosity distance D_L [Mpc]
    z           : float   redshift

    Returns
    -------
    L_v : float   spectral luminosity [erg/s/Hz]
    """
    S_v_cgs  = S_v_Jy * 1e-23                        # Jy  -> erg/s/cm^2/Hz
    D_L_cm   = distance_Mpc * (1.0 * u.Mpc).to(u.cm).value   # Mpc -> cm
    L_v      = (4 * np.pi * D_L_cm**2 / (1 + z)) * S_v_cgs   # erg/s/Hz
    return L_v

In [18]:
def Lv_to_L(L_v, bw_MHz):
    """
    Integrate spectral luminosity over the observing bandwidth.

    Assumes a flat spectrum across the band:
        L = L_v * Delta_nu

    Parameters
    ----------
    L_v    : float   spectral luminosity [erg/s/Hz]
    bw_MHz : float   bandwidth           [MHz]

    Returns
    -------
    L : float   total luminosity [erg/s]
    """
    return L_v * (bw_MHz * 1e6)     # MHz -> Hz

In [25]:
def compute_peak_luminosity(profile, tsamp_s, peak_idx,
                             bw_MHz, freq_ctr_MHz, npol, SEFD,
                             distance_Mpc, z):
    """
    Compute peak luminosity from a (possibly downsampled) time profile.

    Parameters
    ----------
    profile      : 1-D array   normalised time profile (S/N per bin)
    tsamp_s      : float       time resolution of this profile [s]
    peak_idx     : int         bin index of the burst component to measure
                               (pre-scaled to match the current resolution)
    bw_MHz       : float       observing bandwidth [MHz]
    freq_ctr_MHz : float       central frequency of the band [MHz]
    npol         : int         number of polarisations
    SEFD         : float       SEFD [Jy]
    distance_Mpc : float       luminosity distance [Mpc]
    z            : float       redshift

    Returns
    -------
    dict : peak_SN, S_v_Jy, L_v_erg_s_Hz, L_bw_erg_s, L_nu_ctr_erg_s

        L_bw_erg_s     : L_v integrated over the full observing bandwidth
        L_nu_ctr_erg_s : L_v scaled by the central frequency of the band
    """
    peak_SN      = float(profile[peak_idx])
    tsamp_ms     = tsamp_s * 1e3                              # s -> ms
    flux_per_SN  = radiometer(tsamp_ms, bw_MHz, npol, SEFD)  # Jy per S/N unit
    S_v_Jy       = peak_SN * flux_per_SN                      # Jy

    L_v          = flux_to_Lv(S_v_Jy, distance_Mpc, z)       # erg/s/Hz
    L_bw         = Lv_to_L(L_v, bw_MHz)                       # erg/s  (× bandwidth)
    L_nu_ctr     = Lv_to_L(L_v, freq_ctr_MHz)                 # erg/s  (× central freq)

    return dict(peak_SN=peak_SN, S_v_Jy=S_v_Jy,
                L_v_erg_s_Hz=L_v,
                L_bw_erg_s=L_bw,
                L_nu_ctr_erg_s=L_nu_ctr)

In [26]:
def run_peak_luminosity_analysis(
        burst_file,
        burst_tstamp,
        # Telescope / backend
        SEFD             = 420,
        npol             = 2,
        # Source
        distance_Mpc     = 616.0,
        z                = 0.130287,
        # Downsample range
        downsamp_factors = None,   # list of ints; default 1,2,4,...,128
        downsamp_f       = 1,      # frequency downsample factor (fixed)
):
    """
    Full pipeline: load burst at native resolution, then re-measure at
    progressively coarser time resolutions, always tracking the SAME
    burst component.

    The native-resolution (1 us) profile is used once to identify the peak
    bin.  For each coarser resolution the peak bin is simply:

        peak_idx_ds = peak_idx_native // downsamp_factor

    which maps the native sample exactly onto the downsampled grid.

    Parameters
    ----------
    burst_file       : str    path to filterbank file
    burst_tstamp     : float  burst timestamp [s], used to centre time axis
    SEFD             : float  System-Equivalent Flux Density [Jy]
    npol             : int    number of polarisations
    distance_Mpc     : float  luminosity distance [Mpc]
    z                : float  redshift
    downsamp_factors : list   time downsample factors to iterate over
    downsamp_f       : int    frequency downsample factor (fixed)

    Returns
    -------
    results : list of dicts, one per downsample factor, containing:
        downsamp_t, tsamp_us, peak_idx_native, peak_idx_ds,
        peak_SN, S_v_Jy, L_v_erg_s_Hz, L_bw_erg_s, L_nu_ctr_erg_s
    """
    if downsamp_factors is None:
        downsamp_factors = [1, 2, 4, 8, 16, 32, 64, 128]

    # ── Step 1: load native resolution ──────────────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 1 — Loading native-resolution data (downsamp_t=1)")
    print("=" * 60)

    _, prof_native, x_grid, _, extent_b, tsamp_native, fsamp = load_fil(
        burst_file, burst_tstamp, downsamp_t=1, downsamp_f=downsamp_f)

    # x_grid is already relative to burst_tstamp, so burst centre is at 0.0 ms
    contaminated = check_offpulse_contamination(
        x_grid, burst_tstamp_ms=0.0, prof=prof_native,
        sides='right', scale=3, burst_window_ms=50.0)

    if contaminated:
        raise RuntimeError(
            "Off-pulse region is contaminated by the burst window. "
            "Consider using sides='left', or increasing the file length.")

    # Derive band parameters from the extent
    fmin     = extent_b[2]
    fmax     = extent_b[3]
    bw_MHz   = abs(fmax - fmin)
    freq_ctr = (fmin + fmax) / 2.0          # central frequency [MHz]

    print(f"\nBandwidth        : {bw_MHz:.1f} MHz")
    print(f"Central frequency: {freq_ctr:.1f} MHz")
    print(f"Native tsamp     : {tsamp_native * 1e6:.2f} us")

    # Identify peak bin once, at native resolution
    peak_idx_native = int(np.argmax(prof_native))
    print(f"Native peak bin  : {peak_idx_native}  "
          f"(S/N = {prof_native[peak_idx_native]:.2f})")

    results = []

    # ── Step 2: iterate over downsample factors ──────────────────────────────
    for ds in downsamp_factors:
        print("\n" + "=" * 60)
        print(f"downsamp_t = {ds}  ->  tsamp = {tsamp_native * ds * 1e6:.1f} us")
        print("=" * 60)

        if ds == 1:
            # Reuse what we already loaded
            prof_ds  = prof_native
            tsamp_ds = tsamp_native
        else:
            _, prof_ds, _, _, _, tsamp_ds, _ = load_fil(
                burst_file, burst_tstamp,
                downsamp_t=ds, downsamp_f=downsamp_f)

        # Map native peak index onto the coarser grid
        peak_idx_ds = peak_idx_native // ds
        peak_idx_ds = min(peak_idx_ds, len(prof_ds) - 1)   # safety clamp

        print(f"  Peak bin : {peak_idx_native} (native) "
              f"-> {peak_idx_ds} (ds={ds})")

        res = compute_peak_luminosity(
            profile      = prof_ds,
            tsamp_s      = tsamp_ds,
            peak_idx     = peak_idx_ds,
            bw_MHz       = bw_MHz,
            freq_ctr_MHz = freq_ctr,
            npol         = npol,
            SEFD         = SEFD,
            distance_Mpc = distance_Mpc,
            z            = z,
        )
        res.update(dict(
            downsamp_t      = ds,
            tsamp_us        = tsamp_ds * 1e6,
            peak_idx_native = peak_idx_native,
            peak_idx_ds     = peak_idx_ds,
        ))
        results.append(res)

        print(f"  Peak S/N : {res['peak_SN']:.2f}")
        print(f"  S_v      : {res['S_v_Jy']:.4f} Jy")
        print(f"  L_v      : {res['L_v_erg_s_Hz']:.4e} erg/s/Hz")
        print(f"  L (× bw) : {res['L_bw_erg_s']:.4e} erg/s")
        print(f"  L (× nu) : {res['L_nu_ctr_erg_s']:.4e} erg/s")

    # ── Summary table ────────────────────────────────────────────────────────
    print("\n" + "=" * 95)
    print("SUMMARY")
    print(f"  Bandwidth: {bw_MHz:.1f} MHz   |   Central frequency: {freq_ctr:.1f} MHz")
    hdr = (f"{'ds':>4}  {'tsamp(us)':>10}  {'peak S/N':>9}  {'S_v (Jy)':>10}  "
           f"{'L_v (erg/s/Hz)':>16}  {'L×bw (erg/s)':>14}  {'L×nu (erg/s)':>14}")
    print(hdr)
    print("-" * 95)
    for r in results:
        print(f"{r['downsamp_t']:>4}  {r['tsamp_us']:>10.1f}  "
              f"{r['peak_SN']:>9.2f}  {r['S_v_Jy']:>10.4f}  "
              f"{r['L_v_erg_s_Hz']:>16.4e}  "
              f"{r['L_bw_erg_s']:>14.4e}  "
              f"{r['L_nu_ctr_erg_s']:>14.4e}")

    return results

In [37]:
def run_stroop():

    #BURST_FILE   = "/data1/omar/sfxc/frb240114a/p55097_41_wb_60586.95300/p55097_41_8us_500KHz_StokesI_DM528326.cor_Wb.fil"
    BURST_FILE   = "/data1/omar/sfxc/frb240114a/p55097_41_wb_60586.95300/p55097_41_1us_500KHz_StokesI_DM5284.cor_Wb.fil"
    BURST_TSTAMP = 1.079       # seconds — centre of the burst in the file

    results = run_peak_luminosity_analysis(
        burst_file       = BURST_FILE,
        burst_tstamp     = BURST_TSTAMP,
        SEFD             = 420,        # Jy  — Effelsberg L-band
        npol             = 2,
        distance_Mpc     = 616.0,       # Mpc — luminosity distance
        z                = 0.130287,
        downsamp_factors = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024],
        downsamp_f       = 1,
    )

    return
run_stroop()


STEP 1 — Loading native-resolution data (downsamp_t=1)
the extent of the data is: [-1079.0, 1736.9999999999998, 1207.0, 1335.0]
loading in the data
downsample factors are: time:1 and freq:1
Masking subband edges
channel numbers: 256
bandwidth is: 128.0 . Number of channels: 256 . First subband array:  [ 0 31]
Subband edges:  32
  OK: burst window [-25.0, 25.0] ms does not overlap any off-pulse region.

Bandwidth        : 128.0 MHz
Central frequency: 1271.0 MHz
Native tsamp     : 1.00 us
Native peak bin  : 1081416  (S/N = 64.73)

downsamp_t = 1  ->  tsamp = 1.0 us
  Peak bin : 1081416 (native) -> 1081416 (ds=1)
  radiometer inputs:
    SEFD : 420 Jy
    bw   : 128.0 MHz
    npol : 2
    tsamp: 0.001 ms
  Peak S/N : 64.73
  S_v      : 1699.0570 Jy
  L_v      : 6.8248e+35 erg/s/Hz
  L (× bw) : 8.7358e+43 erg/s
  L (× nu) : 8.6744e+44 erg/s

downsamp_t = 2  ->  tsamp = 2.0 us
the extent of the data is: [-1079.0, 1736.9999999999998, 1207.0, 1335.0]
loading in the data
downsample factors ar